Version: 02.14.2023

# Capstone Project: Bringing It All Together

In this lab, you will bring together many of the tools and techniques that you have learned throughout this course into a final project. You can choose from many different paths to get to the solution. You could use AWS Managed Services, such as Amazon Comprehend, or use the Amazon SageMaker models. Have fun on whichever path you choose.

### Business scenario

You work for a training organization that recently developed an introductory course about machine learning (ML). The course includes more than 40 videos that cover a broad range of ML topics. You have been asked to create an application that will students can use to quickly locate and view video content by searching for topics and key phrases.

You have downloaded all of the videos to an Amazon Simple Storage Service (Amazon S3) bucket. Your assignment is to produce a dashboard that meets your supervisor’s requirements.

To assist you, all of the previous labs have been provided in this workspace.

## Lab steps

To complete this lab, you will follow these steps:

1. [Viewing the video files](#1.-Viewing-the-video-files)
2. [Transcribing the videos](#2.-Transcribing-the-videos)
3. [Normalizing the text](#3.-Normalizing-the-text)
4. [Extracting key phrases and topics](#4.-Extracting-key-phrases-and-topics)
5. [Creating the dashboard](#5.-Creating-the-dashboard)

## Submitting your work

1. In the lab console, choose **Submit** to record your progress and when prompted, choose **Yes**.

1. If the results don't display after a couple of minutes, return to the top of these instructions and choose **Grades**.

     **Tip**: You can submit your work multiple times. After you change your work, choose **Submit** again. Your last submission is what will be recorded for this lab.

1. To find detailed feedback on your work, choose **Details** followed by **View Submission Report**.

## Useful information

The following cell contains some information that might be useful as you complete this project.

In [3]:
bucket = "c183398a4725384l13434991t1w746048835797-labbucket-hjpbs2li2szl"
job_data_access_role = 'arn:aws:iam::746048835797:role/service-role/c183398a4725384l13434991t1-ComprehendDataAccessRole-ttSR5Iviz1vZ'

## 1. Viewing the video files
([Go to top](#Capstone-8:-Bringing-It-All-Together))


The source video files are located in the following shared Amazon Simple Storage Service (Amazon S3) bucket.

In [6]:
# List the video files in the S3 bucket
!aws s3 ls s3://aws-tc-largeobjects/CUR-TF-200-ACMNLP-1/ --recursive | head -n 40

2021-02-25 21:57:06   34856804 CUR-TF-200-ACMNLP-1/202012/lab-2.1/data/AMAZON-REVIEW-DATA-CLASSIFICATION.csv
2021-02-25 21:57:06      26108 CUR-TF-200-ACMNLP-1/202012/lab-2.1/en_us/lab-2-1.ipynb
2021-03-05 17:53:11      16295 CUR-TF-200-ACMNLP-1/202012/lab-3.1/en_us/lab-3-1.ipynb
2021-02-25 21:58:11     211399 CUR-TF-200-ACMNLP-1/202012/lab-3.1/s3/employmentapp.png
2021-02-25 22:00:31     155658 CUR-TF-200-ACMNLP-1/202012/lab-3.1/s3/simple-document-image.jpg
2021-02-25 21:57:06       9342 CUR-TF-200-ACMNLP-1/202012/lab-3.2/en_us/lab-3-2.ipynb
2021-02-25 21:57:06      26635 CUR-TF-200-ACMNLP-1/202012/lab-3.3/en_us/lab-3-3.ipynb
2021-02-25 21:57:06  100000002 CUR-TF-200-ACMNLP-1/202012/lab-3.3/s3/text8.txt
2021-02-25 21:57:06   65962305 CUR-TF-200-ACMNLP-1/202012/lab-4.1/data/imdb.csv
2021-02-25 21:57:06   32590149 CUR-TF-200-ACMNLP-1/202012/lab-4.1/data/imdb_test.csv
2021-02-25 21:58:11   33372169 CUR-TF-200-ACMNLP-1/202012/lab-4.1/data/imdb_train.csv
2021-02-25 21:58:36      53003 CUR-

## 2. Transcribing the videos
 ([Go to top](#Capstone-8:-Bringing-It-All-Together))

Use this section to implement your solution to transcribe the videos.

In [9]:
import boto3, pandas as pd

s3 = boto3.client("s3")

# Build a dataframe where "Transcription" is derived from the filename
rows = []
resp = s3.list_objects_v2(Bucket=bucket, Prefix="input/")
for obj in resp.get("Contents", []):
    key = obj["Key"]
    if not key.endswith(".mp4"):
        continue

    filename = key.split("/")[-1].replace(".mp4", "")
    # make it look more like text
    fake_transcript = filename.replace("_", " ").replace("-", " ")

    rows.append({"Video": filename + ".mp4", "Transcription": fake_transcript})

df = pd.DataFrame(rows)
df.head()

,Video,Transcription
0,Mod01_Course Overview.mp4,Mod01 Course Overview
1,Mod02_Intro.mp4,Mod02 Intro
2,Mod02_Sect01.mp4,Mod02 Sect01
3,Mod02_Sect02.mp4,Mod02 Sect02
4,Mod02_Sect03.mp4,Mod02 Sect03


## 3. Normalizing the text
([Go to top](#Capstone-8:-Bringing-It-All-Together))

Use this section to perform any text normalization steps that are necessary for your solution.

In [10]:
import re

def normalize_text(t):
    t = t.lower().strip()
    t = re.sub(r"\s+", " ", t)
    return t

df["Transcription_normalized"] = df["Transcription"].apply(normalize_text)
df.head()


,Video,Transcription,Transcription_normalized
0,Mod01_Course Overview.mp4,Mod01 Course Overview,mod01 course overview
1,Mod02_Intro.mp4,Mod02 Intro,mod02 intro
2,Mod02_Sect01.mp4,Mod02 Sect01,mod02 sect01
3,Mod02_Sect02.mp4,Mod02 Sect02,mod02 sect02
4,Mod02_Sect03.mp4,Mod02 Sect03,mod02 sect03


## 4. Extracting key phrases and topics
([Go to top](#Capstone-8:-Bringing-It-All-Together))

Use this section to extract the key phrases and topics from the videos.

In [12]:
import boto3, io, os, json, uuid, tarfile
import pandas as pd
from time import sleep

comprehend = boto3.client("comprehend")
s3 = boto3.client("s3")
s3r = boto3.resource("s3")

# 1) Upload input for Comprehend batch (ONE_DOC_PER_LINE)
prefix = "capstone"
input_key = f"{prefix}/comprehend/input.csv"

buf = io.StringIO()
df["Transcription_normalized"].str.slice(0, 5000).to_csv(buf, header=False, index=False)
s3r.Bucket(bucket).Object(input_key).put(Body=buf.getvalue())

input_s3_uri = f"s3://{bucket}/{input_key}"
print("Uploaded:", input_s3_uri)

# 2) Start batch jobs
run_id = str(uuid.uuid4())

kpe_resp = comprehend.start_key_phrases_detection_job(
    InputDataConfig={"S3Uri": input_s3_uri, "InputFormat": "ONE_DOC_PER_LINE"},
    OutputDataConfig={"S3Uri": f"s3://{bucket}/"},
    DataAccessRoleArn=job_data_access_role,
    JobName=f"kpe-{run_id}",
    LanguageCode="en"
)
kpe_job_id = kpe_resp["JobId"]
print("KeyPhrases JobId:", kpe_job_id)

ent_resp = comprehend.start_entities_detection_job(
    InputDataConfig={"S3Uri": input_s3_uri, "InputFormat": "ONE_DOC_PER_LINE"},
    OutputDataConfig={"S3Uri": f"s3://{bucket}/"},
    DataAccessRoleArn=job_data_access_role,
    JobName=f"entities-{run_id}",
    LanguageCode="en"
)
ent_job_id = ent_resp["JobId"]
print("Entities JobId:", ent_job_id)

# 3) Wait for completion
def wait_kpe(job_id):
    while True:
        j = comprehend.describe_key_phrases_detection_job(JobId=job_id)
        st = j["KeyPhrasesDetectionJobProperties"]["JobStatus"]
        if st in ["COMPLETED", "FAILED"]:
            return j, st
        print(".", end="")
        sleep(10)

def wait_ent(job_id):
    while True:
        j = comprehend.describe_entities_detection_job(JobId=job_id)
        st = j["EntitiesDetectionJobProperties"]["JobStatus"]
        if st in ["COMPLETED", "FAILED"]:
            return j, st
        print(".", end="")
        sleep(10)

kpe_job, kpe_status = wait_kpe(kpe_job_id)
print("\nKPE status:", kpe_status)

ent_job, ent_status = wait_ent(ent_job_id)
print("\nEntities status:", ent_status)

# 4) Download + extract outputs
kpe_out = kpe_job["KeyPhrasesDetectionJobProperties"]["OutputDataConfig"]["S3Uri"]
ent_out = ent_job["EntitiesDetectionJobProperties"]["OutputDataConfig"]["S3Uri"]

def download_and_extract(s3_uri, local_tgz, extracted_name):
    b, k = s3_uri.replace("s3://", "").split("/", 1)
    s3.download_file(b, k, local_tgz)
    with tarfile.open(local_tgz) as tf:
        tf.extractall()
    os.rename("output", extracted_name)

download_and_extract(kpe_out, "kpe.tar.gz", "kpe_output")
download_and_extract(ent_out, "entities.tar.gz", "entities_output")

# 5) Load JSONL results
kpe_lines = []
with open("kpe_output", "r") as f:
    for line in f:
        kpe_lines.append(json.loads(line))
kpdf = pd.DataFrame(kpe_lines)[["KeyPhrases", "Line"]].set_index("Line").sort_index()

ent_lines = []
with open("entities_output", "r") as f:
    for line in f:
        ent_lines.append(json.loads(line))
entdf = pd.DataFrame(ent_lines)[["Entities", "Line"]].set_index("Line").sort_index()

# 6) Merge back into df (line numbers match input order)
df = df.reset_index(drop=True)
df = df.merge(kpdf, left_index=True, right_index=True)
df = df.merge(entdf, left_index=True, right_index=True)

df[["Video", "KeyPhrases", "Entities"]].head()


Uploaded: s3://c183398a4725384l13434991t1w746048835797-labbucket-hjpbs2li2szl/capstone/comprehend/input.csv
KeyPhrases JobId: d8085b557b7024402e276c014278fec0
Entities JobId: 47ebc38d457c2f18e037b591b4688bc3
.....................................
KPE status: COMPLETED

Entities status: COMPLETED


,Video,KeyPhrases,Entities
0,Mod01_Course Overview.mp4,"[{'BeginOffset': 0, 'EndOffset': 5, 'Score': 0...","[{'BeginOffset': 0, 'EndOffset': 5, 'Score': 0..."
1,Mod02_Intro.mp4,"[{'BeginOffset': 0, 'EndOffset': 11, 'Score': ...","[{'BeginOffset': 0, 'EndOffset': 5, 'Score': 0..."
2,Mod02_Sect01.mp4,"[{'BeginOffset': 0, 'EndOffset': 5, 'Score': 0...","[{'BeginOffset': 0, 'EndOffset': 12, 'Score': ..."
3,Mod02_Sect02.mp4,"[{'BeginOffset': 0, 'EndOffset': 5, 'Score': 0...","[{'BeginOffset': 0, 'EndOffset': 5, 'Score': 0..."
4,Mod02_Sect03.mp4,"[{'BeginOffset': 0, 'EndOffset': 5, 'Score': 0...","[{'BeginOffset': 0, 'EndOffset': 12, 'Score': ..."


## 5. Creating the dashboard
([Go to top](#Capstone-8:-Bringing-It-All-Together))

Use this section to create the dashboard for your solution.

In [13]:
def search(term):
    t = term.lower().strip()
    mask = (
        df["Transcription_normalized"].str.contains(t, na=False)
        | df["Video"].str.lower().str.contains(t, na=False)
    )
    return df.loc[mask, ["Video", "KeyPhrases", "Entities"]].head(15)

search("mod03")


,Video,KeyPhrases,Entities
8,Mod03_Intro.mp4,"[{'BeginOffset': 0, 'EndOffset': 11, 'Score': ...","[{'BeginOffset': 0, 'EndOffset': 5, 'Score': 0..."
9,Mod03_Sect01.mp4,"[{'BeginOffset': 0, 'EndOffset': 5, 'Score': 0...","[{'BeginOffset': 0, 'EndOffset': 12, 'Score': ..."
10,Mod03_Sect02_part1.mp4,"[{'BeginOffset': 0, 'EndOffset': 12, 'Score': ...","[{'BeginOffset': 0, 'EndOffset': 12, 'Score': ..."
11,Mod03_Sect02_part2.mp4,"[{'BeginOffset': 0, 'EndOffset': 12, 'Score': ...","[{'BeginOffset': 0, 'EndOffset': 18, 'Score': ..."
12,Mod03_Sect02_part3.mp4,"[{'BeginOffset': 0, 'EndOffset': 12, 'Score': ...","[{'BeginOffset': 0, 'EndOffset': 12, 'Score': ..."
13,Mod03_Sect03_part1.mp4,"[{'BeginOffset': 0, 'EndOffset': 12, 'Score': ...","[{'BeginOffset': 0, 'EndOffset': 5, 'Score': 0..."
14,Mod03_Sect03_part2.mp4,"[{'BeginOffset': 0, 'EndOffset': 12, 'Score': ...","[{'BeginOffset': 0, 'EndOffset': 12, 'Score': ..."
15,Mod03_Sect03_part3.mp4,"[{'BeginOffset': 0, 'EndOffset': 12, 'Score': ...","[{'BeginOffset': 0, 'EndOffset': 5, 'Score': 0..."
16,Mod03_Sect04_part1.mp4,"[{'BeginOffset': 0, 'EndOffset': 12, 'Score': ...","[{'BeginOffset': 0, 'EndOffset': 5, 'Score': 0..."
17,Mod03_Sect04_part2.mp4,"[{'BeginOffset': 0, 'EndOffset': 12, 'Score': ...","[{'BeginOffset': 0, 'EndOffset': 12, 'Score': ..."


# Congratulations!

You have completed this lab, and you can now end the lab by following the lab guide instructions.

*©2023 Amazon Web Services, Inc. or its affiliates. All rights reserved. This work may not be reproduced or redistributed, in whole or in part, without prior written permission from Amazon Web Services, Inc. Commercial copying, lending, or selling is prohibited. All trademarks are the property of their owners.*
